In [24]:
import os
import sys
import torch
from torch.utils.data import DataLoader
from torch import nn
from torchvision import datasets,transforms,models
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [25]:
# List all datasets mounted in Kaggle
print("Datasets available in /kaggle/input/:")
for i in os.listdir("/kaggle/input/datasets/kaushikrajagiri/plant-diesease-dataset"):
    print(i)


Datasets available in /kaggle/input/:
Leaf-miner
Gummosis
Canker
Healthy
Lemon-butterfly
Greening


In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),   
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

base_dataset = datasets.ImageFolder(root = "/kaggle/input/datasets/kaushikrajagiri/plant-diesease-dataset", transform=None)
train_size = int(0.8 * len(base_dataset))
test_size = len(base_dataset) - train_size
indices = torch.randperm(len(base_dataset)).tolist()

class SubsetWithTransform(torch.utils.data.Subset):
    def __init__(self, dataset, indices, transform=None):
        super().__init__(dataset, indices)
        self.transform = transform

    def __getitem__(self, idx):
        image, label = super().__getitem__(idx)
        if self.transform is not None:
            image = self.transform(image)
        return image, label

train_dataset = SubsetWithTransform(base_dataset, indices[:train_size], transform=train_transform)
test_dataset = SubsetWithTransform(base_dataset, indices[train_size:], transform=test_transform)

In [27]:
img_tensor,label = train_dataset[0]
print(img_tensor.shape)

torch.Size([3, 224, 224])


In [28]:
import torchvision.transforms as transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),   
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


In [29]:
class Resnet(nn.Module):
    def __init__(self,output_class:int):
        super().__init__()
        self.resnet = models.resnet18(pretrained = True)
        in_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Linear(in_features,output_class)

    def forward(self,x):
        return self.resnet(x)

In [30]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = Resnet(output_class = 6).to(device)
model

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Resnet(
  (resnet): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_runni

In [31]:
for param in model.resnet.parameters():
    param.requires_grad = False

In [32]:
for param in model.resnet.fc.parameters():
    param.requires_grad = True

## loss function and optimizer

In [33]:
loss_func = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.resnet.fc.parameters(),lr= 0.01)

## training loop

In [34]:
epochs = 3
for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    correct = 0
    total = 0
    for images, labels in DataLoader(train_dataset, batch_size=32, shuffle=True):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_func(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
    epoch_loss = train_loss / len(train_dataset)
    epoch_acc = correct / total
    print(f"Epoch [{epoch+1}/{epochs}] "
          f"Loss: {epoch_loss:.4f} "
          f"Accuracy: {epoch_acc:.4f}")


Epoch [1/3] Loss: 0.2176 Accuracy: 0.9249
Epoch [2/3] Loss: 0.1105 Accuracy: 0.9627
Epoch [3/3] Loss: 0.1021 Accuracy: 0.9694


In [35]:
def accuracy_function(y_true, y_pred):
    y_pred_tags = torch.argmax(y_pred, dim=1)
    correct_results_sum = (y_pred_tags == y_true).sum().item()
    return correct_results_sum

## testing loop

In [36]:
test_loss = 0.0
test_acc =0
model.eval()
with torch.inference_mode():
    for images_test,labels_test in DataLoader(test_dataset,batch_size=32,shuffle=True):
        images_test,labels_test = images_test.to(device),labels_test.to(device)
        test_outputs= model(images_test)
        test_loss += loss_func(test_outputs,labels_test).item()*images_test.size(0)
        test_acc += accuracy_function(labels_test,test_outputs)

    test_loss /=len(test_dataset)  
    test_acc /= len(test_dataset)

print(f"test_loss: {test_loss:.4f} | test_acc: {test_acc:.2f}%")


test_loss: 0.0746 | test_acc: 0.98%


In [37]:
print(train_dataset.dataset.classes)
print(test_dataset.dataset.classes)

print(train_dataset.dataset.class_to_idx)
print(test_dataset.dataset.class_to_idx)


['Canker', 'Greening', 'Gummosis', 'Healthy', 'Leaf-miner', 'Lemon-butterfly']
['Canker', 'Greening', 'Gummosis', 'Healthy', 'Leaf-miner', 'Lemon-butterfly']
{'Canker': 0, 'Greening': 1, 'Gummosis': 2, 'Healthy': 3, 'Leaf-miner': 4, 'Lemon-butterfly': 5}
{'Canker': 0, 'Greening': 1, 'Gummosis': 2, 'Healthy': 3, 'Leaf-miner': 4, 'Lemon-butterfly': 5}
